In [1]:
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os

load_dotenv()
engine = create_engine(os.getenv("DATABASE_URL"))

df = pd.read_sql("SELECT * FROM matches WHERE status = 'FINISHED'", engine)
print(df.shape)
df.head()

(836, 10)


,id,match_id,home_team,away_team,home_score,away_score,status,match_date,stage,season
0,1,1096,France,Mexico,4,1,FINISHED,1930-07-13 15:00:00,Group 1,1930
1,2,1090,USA,Belgium,3,0,FINISHED,1930-07-13 15:00:00,Group 4,1930
2,3,1093,Yugoslavia,Brazil,2,1,FINISHED,1930-07-14 12:45:00,Group 2,1930
3,4,1098,Romania,Peru,3,1,FINISHED,1930-07-14 14:50:00,Group 3,1930
4,5,1085,Argentina,France,1,0,FINISHED,1930-07-15 16:00:00,Group 1,1930


In [2]:
# Resultado: 0=local gana, 1=empate, 2=visitante gana
def get_result(row):
    if row["home_score"] > row["away_score"]:
        return 0
    elif row["home_score"] == row["away_score"]:
        return 1
    else:
        return 2

df["result"] = df.apply(get_result, axis=1)

# Codificar equipos
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df["home_encoded"] = le.fit_transform(df["home_team"])
df["away_encoded"] = le.fit_transform(df["away_team"])

# Stage encoding
stage_map = {
    "Group Stage": 1, "Round of 16": 2, "Quarter-finals": 3,
    "Semi-finals": 4, "Third place": 5, "Final": 6
}
df["stage_encoded"] = df["stage"].map(stage_map).fillna(1)

features = ["home_encoded", "away_encoded", "stage_encoded", "season"]
X = df[features]
y = df["result"]
print(X.shape, y.value_counts())

(836, 4) result
0    479
1    186
2    171
Name: count, dtype: int64


In [4]:
import os
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import pickle

os.makedirs("../models", exist_ok=True)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = XGBClassifier(n_estimators=100, max_depth=4, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(f"✅ Accuracy: {accuracy_score(y_test, y_pred):.2f}")
print(classification_report(y_test, y_pred, target_names=["Local", "Empate", "Visitante"]))

with open("../models/model.pkl", "wb") as f:
    pickle.dump(model, f)
with open("../models/label_encoder.pkl", "wb") as f:
    pickle.dump(le, f)

print("✅ Modelo guardado en models/")

✅ Accuracy: 0.51
              precision    recall  f1-score   support

       Local       0.60      0.72      0.65        92
      Empate       0.24      0.15      0.18        41
   Visitante       0.42      0.40      0.41        35

    accuracy                           0.51       168
   macro avg       0.42      0.42      0.42       168
weighted avg       0.48      0.51      0.49       168

✅ Modelo guardado en models/


In [5]:
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os

load_dotenv()
engine = create_engine(os.getenv("DATABASE_URL"))

df = pd.read_sql("SELECT * FROM matches WHERE status = 'FINISHED'", engine)
stats = pd.read_sql("SELECT * FROM team_stats", engine)
print(f"Partidos: {len(df)} | Stats: {len(stats)}")

Partidos: 836 | Stats: 427


In [6]:
def get_result(row):
    if row["home_score"] > row["away_score"]: return 0
    elif row["home_score"] == row["away_score"]: return 1
    else: return 2

df["result"] = df.apply(get_result, axis=1)

# Merge estadísticas del equipo local
df = df.merge(stats, left_on=["home_team", "season"], right_on=["team", "season"], how="left", suffixes=("", "_home"))
df = df.rename(columns={"wins": "home_wins", "draws": "home_draws", "losses": "home_losses",
                          "goals_for": "home_goals_for", "goals_against": "home_goals_against",
                          "matches_played": "home_matches_played"})

# Merge estadísticas del equipo visitante
df = df.merge(stats, left_on=["away_team", "season"], right_on=["team", "season"], how="left", suffixes=("", "_away"))
df = df.rename(columns={"wins": "away_wins", "draws": "away_draws", "losses": "away_losses",
                          "goals_for": "away_goals_for", "goals_against": "away_goals_against",
                          "matches_played": "away_matches_played"})

# Encodear equipos
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
all_teams = pd.concat([df["home_team"], df["away_team"]]).unique()
le.fit(all_teams)
df["home_encoded"] = le.transform(df["home_team"])
df["away_encoded"] = le.transform(df["away_team"])

# Stage encoding
stage_map = {"Group Stage": 1, "Round of 16": 2, "Quarter-finals": 3,
             "Semi-finals": 4, "Third place": 5, "Final": 6}
df["stage_encoded"] = df["stage"].map(stage_map).fillna(1)

# Win rate
df["home_win_rate"] = df["home_wins"] / (df["home_matches_played"] + 1)
df["away_win_rate"] = df["away_wins"] / (df["away_matches_played"] + 1)
df["home_goal_diff"] = df["home_goals_for"] - df["home_goals_against"]
df["away_goal_diff"] = df["away_goals_for"] - df["away_goals_against"]

features = ["home_encoded", "away_encoded", "stage_encoded", "season",
            "home_win_rate", "away_win_rate", "home_goal_diff", "away_goal_diff",
            "home_wins", "away_wins", "home_draws", "away_draws"]

X = df[features].fillna(0)
y = df["result"]
print(f"Features: {len(features)} | Shape: {X.shape}")
print(y.value_counts())

Features: 12 | Shape: (836, 12)
result
0    479
1    186
2    171
Name: count, dtype: int64


In [7]:
import os
import pickle
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

os.makedirs("../models", exist_ok=True)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.05, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(f"✅ Accuracy: {accuracy_score(y_test, y_pred):.2f}")
print(classification_report(y_test, y_pred, target_names=["Local", "Empate", "Visitante"]))

with open("../models/model.pkl", "wb") as f:
    pickle.dump(model, f)
with open("../models/label_encoder.pkl", "wb") as f:
    pickle.dump(le, f)

# Guardar lista de features para usarla en la API
with open("../models/features.pkl", "wb") as f:
    pickle.dump(features, f)

print("✅ Modelo mejorado guardado en models/")

✅ Accuracy: 0.77
              precision    recall  f1-score   support

       Local       0.84      0.89      0.86        92
      Empate       0.62      0.56      0.59        41
   Visitante       0.73      0.69      0.71        35

    accuracy                           0.77       168
   macro avg       0.73      0.71      0.72       168
weighted avg       0.76      0.77      0.76       168

✅ Modelo mejorado guardado en models/
